Name:- Tanmay Santosh Sanap

Roll No:-40

Student Id:- 5144978

College:- B K BIRLA COLLEGE

Subject:-Data Engineering (Practical No :- 09)

To design and implement an end-to-end data pipeline that extracts e-commerce data from CSV files, cleans and transforms the data using Python/Pandas, loads it into a SQL database, generates a report, and handles both historical and newly arriving data.

In [2]:
import pandas as pd
import sqlite3
import os

print("=" * 65)
print("              PRACTICAL 9 - ETL PIPELINE")
print("=" * 65)


# ============================================================
# TASK 1
# END-TO-END CSV -> PANDAS -> CLEANING -> SQL -> REPORT
# ============================================================

print("\n\n========== TASK 1 ==========")
print("End-to-End CSV to SQL Pipeline")


csv_file = "raw_sales.csv"
db_file = "task1_pipeline.db"


# ------------------------------------------------------------
# 1. SETUP: Create messy raw data
# ------------------------------------------------------------

with open(csv_file, "w") as f:

    f.write("transaction_id,date,amount,category\n")

    f.write("1,2026-10-01,150.50,Electronics\n")

    f.write("2,,200.00,Clothing\n")       # Missing date

    f.write("3,2026-10-03,ERROR,Electronics\n")  # Invalid amount

    f.write("4,2026-10-04,99.99,Clothing\n")


# ------------------------------------------------------------
# 2. EXTRACT: Read CSV using Pandas
# ------------------------------------------------------------

df = pd.read_csv(csv_file)

print("\nOriginal Data:")
print(df)


# ------------------------------------------------------------
# 3. DATA CLEANING
# ------------------------------------------------------------

# Remove rows where date is missing
df = df.dropna(subset=["date"])


# Convert amount to numeric
# Invalid values become NaN
df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)


# Remove rows where amount is invalid
df = df.dropna(subset=["amount"])


# Format date
df["date"] = pd.to_datetime(
    df["date"]
).dt.strftime("%Y-%m-%d")


print("\nCleaned Data:")
print(df)


# ------------------------------------------------------------
# 4. LOAD INTO SQL DATABASE
# ------------------------------------------------------------

conn = sqlite3.connect(db_file)

df.to_sql(
    "cleaned_sales",
    conn,
    if_exists="replace",
    index=False
)


# ------------------------------------------------------------
# 5. GENERATE REPORT USING SQL
# ------------------------------------------------------------

report_query = """
SELECT
    category,
    COUNT(transaction_id) AS total_sales,
    SUM(amount) AS total_revenue
FROM cleaned_sales
GROUP BY category
"""

report_df = pd.read_sql(
    report_query,
    conn
)


print("\nFinal Aggregated Report:")
print(
    report_df.to_string(index=False)
)


conn.close()


# ============================================================
# TASK 2
# E-COMMERCE ETL PIPELINE
# ============================================================

print("\n\n========== TASK 2 ==========")
print("E-Commerce ETL Pipeline")


db_file = "task2_ecommerce.db"


# ------------------------------------------------------------
# 1. EXTRACT
# Create Customers Data
# ------------------------------------------------------------

customers = pd.DataFrame({
    "cust_id": [1, 2],
    "name": ["Alice", "Bob"]
})


# ------------------------------------------------------------
# Products Data
# ------------------------------------------------------------

products = pd.DataFrame({
    "prod_id": [101, 102],
    "price": [1200.0, 50.0]
})


# ------------------------------------------------------------
# Orders Data
# ------------------------------------------------------------

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "cust_id": [1, 1, 2],
    "prod_id": [101, 102, 102],
    "qty": [1, 2, 1]
})


# ------------------------------------------------------------
# Payments Data
# ------------------------------------------------------------

payments = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "status": [
        "Completed",
        "Failed",
        "Completed"
    ]
})


print("\nCustomers:")
print(customers)

print("\nProducts:")
print(products)

print("\nOrders:")
print(orders)

print("\nPayments:")
print(payments)


# ------------------------------------------------------------
# 2. TRANSFORM
# ------------------------------------------------------------

# Join Orders and Payments
merged = orders.merge(
    payments,
    on="order_id"
)


# Keep only successful payments
successful_orders = merged[
    merged["status"] == "Completed"
]


# Join with Products
successful_orders = successful_orders.merge(
    products,
    on="prod_id"
)


# Calculate total transaction value
successful_orders["total_value"] = (
    successful_orders["qty"] *
    successful_orders["price"]
)


# Join with Customers
final_merged = successful_orders.merge(
    customers,
    on="cust_id"
)


# Calculate customer lifetime revenue
customer_revenue = (
    final_merged
    .groupby("name")["total_value"]
    .sum()
    .reset_index()
)


# Rename column
customer_revenue.rename(
    columns={
        "total_value": "lifetime_revenue"
    },
    inplace=True
)


# ------------------------------------------------------------
# 3. LOAD INTO SQL DATABASE
# ------------------------------------------------------------

conn = sqlite3.connect(db_file)

customer_revenue.to_sql(
    "customer_revenue_report",
    conn,
    if_exists="replace",
    index=False
)


# ------------------------------------------------------------
# 4. DISPLAY REPORT
# ------------------------------------------------------------

print(
    "\nFinal Customer Revenue Table:"
)

print(
    pd.read_sql(
        "SELECT * FROM customer_revenue_report",
        conn
    ).to_string(index=False)
)


conn.close()


# ============================================================
# TASK 3
# HISTORICAL + NEWLY ARRIVING DATA
# ============================================================

print("\n\n========== TASK 3 ==========")
print("Historical and Newly Arriving Data Pipeline")


db_file = "task3_incremental.db"


# ------------------------------------------------------------
# 1. CREATE DATABASE
# ------------------------------------------------------------

conn = sqlite3.connect(db_file)

cursor = conn.cursor()


# ------------------------------------------------------------
# CREATE TABLE FOR HISTORICAL DATA
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE user_profiles (
    user_id INTEGER PRIMARY KEY,
    email TEXT,
    status TEXT
)
""")


# ------------------------------------------------------------
# HISTORICAL DATA
# ------------------------------------------------------------

historical_data = [
    (1, "alice@test.com", "Active"),
    (2, "bob@test.com", "Active")
]


cursor.executemany(
    """
    INSERT INTO user_profiles
    VALUES (?, ?, ?)
    """,
    historical_data
)


conn.commit()


# ------------------------------------------------------------
# DISPLAY HISTORICAL DATA
# ------------------------------------------------------------

print("\nInitial Historical Data in DB:")

print(
    pd.read_sql(
        "SELECT * FROM user_profiles",
        conn
    ).to_string(index=False)
)


# ------------------------------------------------------------
# 2. NEWLY ARRIVING DATA
# ------------------------------------------------------------

new_data_df = pd.DataFrame({
    "user_id": [2, 3],
    "email": [
        "bob_updated@test.com",
        "charlie@test.com"
    ],
    "status": [
        "Inactive",
        "Active"
    ]
})


print("\nNewly Arriving Data:")

print(new_data_df)


# ------------------------------------------------------------
# 3. INCREMENTAL LOAD
# ------------------------------------------------------------

# Step A:
# Load new data into staging table

new_data_df.to_sql(
    "staging_profiles",
    conn,
    if_exists="replace",
    index=False
)


# Step B:
# UPSERT data into main table

cursor.execute("""
INSERT OR REPLACE INTO user_profiles
(
    user_id,
    email,
    status
)
SELECT
    user_id,
    email,
    status
FROM staging_profiles
""")


conn.commit()


# Step C:
# Remove staging table

cursor.execute(
    "DROP TABLE staging_profiles"
)

conn.commit()


# ------------------------------------------------------------
# 4. DISPLAY FINAL DATABASE
# ------------------------------------------------------------

print(
    "\nDatabase State After Incremental Load:"
)

print(
    pd.read_sql(
        "SELECT * FROM user_profiles",
        conn
    ).to_string(index=False)
)


# ------------------------------------------------------------
# CLOSE DATABASE
# ------------------------------------------------------------

conn.close()


# ============================================================
# CLEANUP
# ============================================================

if os.path.exists(csv_file):
    os.remove(csv_file)

if os.path.exists("task1_pipeline.db"):
    os.remove("task1_pipeline.db")

if os.path.exists("task2_ecommerce.db"):
    os.remove("task2_ecommerce.db")

if os.path.exists("task3_incremental.db"):
    os.remove("task3_incremental.db")


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 65)
print("             ALL PRACTICAL 9 TASKS COMPLETED")
print("=" * 65)

              PRACTICAL 9 - ETL PIPELINE


========== TASK 1 ==========
End-to-End CSV to SQL Pipeline

Original Data:
   transaction_id        date  amount     category
0               1  2026-10-01  150.50  Electronics
1               2         NaN  200.00     Clothing
2               3  2026-10-03   ERROR  Electronics
3               4  2026-10-04   99.99     Clothing

Cleaned Data:
   transaction_id        date  amount     category
0               1  2026-10-01  150.50  Electronics
3               4  2026-10-04   99.99     Clothing

Final Aggregated Report:
   category  total_sales  total_revenue
   Clothing            1          99.99
Electronics            1         150.50


========== TASK 2 ==========
E-Commerce ETL Pipeline

Customers:
   cust_id   name
0        1  Alice
1        2    Bob

Products:
   prod_id   price
0      101  1200.0
1      102    50.0

Orders:
   order_id  cust_id  prod_id  qty
0      1001        1      101    1
1      1002        1      102    2
2      10